# Movie Recommender System
This notebook demonstrates multiple recommendation system techniques using the MovieLens 100K dataset.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")


## 📥 Load and Preprocess Data

In [24]:
# Load data
movies = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/movies.csv')
ratings = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/ratings.csv')
tags = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/tags.csv')

# Convert timestamps
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit='s')

# Merge tags with movies
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ','.join(x)).reset_index()
movies = pd.merge(movies, tags_grouped, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')
movies['combined_features'] = movies['genres'] + ' ' + movies['tag']

# Preview data
movies.head()


,movieId,title,genres,tag,combined_features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"pixar,pixar,fun",Adventure|Animation|Children|Comedy|Fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,"fantasy,magic board game,Robin Williams,game","Adventure|Children|Fantasy fantasy,magic board..."
2,3,Grumpier Old Men (1995),Comedy|Romance,"moldy,old","Comedy|Romance moldy,old"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,"pregnancy,remake","Comedy pregnancy,remake"


In [7]:
# Load data
movies = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/movies.csv')
ratings = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/ratings.csv')
tags = pd.read_csv('/Users/omarcharif/Downloads/ml-latest-small/tags.csv')

In [4]:
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
movies_ratings = pd.merge(ratings, movies, left_on="movieId",right_on="movieId",how='left')
movies_ratings

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...
100831,610,166534,4.0,2017-05-03 21:53:22,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,2017-05-03 22:21:31,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,2017-05-08 19:50:47,Get Out (2017),Horror
100834,610,168252,5.0,2017-05-03 21:19:12,Logan (2017),Action|Sci-Fi


In [ ]:
weight_rating = 1/(1 + number_of_days_passed)

In [88]:
grouped_by_movies = movies_ratings.groupby(["movieId", "title"])["rating"].mean().reset_index().sort_values(by="rating", ascending=False).reset_index()
grouped_by_movies

,index,movieId,title,rating
0,7638,88448,Paper Birds (Pájaros de papel) (2010),5.0
1,8089,100556,"Act of Killing, The (2012)",5.0
2,9065,143031,Jump In! (2007),5.0
3,9076,143511,Human (2015),5.0
4,9078,143559,L.A. Slasher (2015),5.0
...,...,...,...,...
9719,9253,157172,Wizards of the Lost Kingdom II (1989),0.5
9720,7536,85334,Hard Ticket to Hawaii (1987),0.5
9721,6486,53453,Starcrash (a.k.a. Star Crash) (1978),0.5
9722,5200,8494,"Cincinnati Kid, The (1965)",0.5


In [76]:
ratings.shape

(100836, 4)

In [77]:
movies.shape

(9742, 3)

In [75]:
movies_ratings.shape

(100836, 6)

In [78]:
movies_ratings.loc[movies_ratings["movieId"] == 147330,:]

,userId,movieId,rating,timestamp,title,genres
16892,105,147330,5.0,2018-05-13 10:30:03,Sherlock Holmes and Dr. Watson: Acquaintance (...,Crime


In [86]:
ordered_grouped_movies_rating = movies_ratings.groupby("movieId").agg({
        "rating": ["mean", "max", "count"],
        "title": "first"
    }
).sort_values(by=("rating", "count"), ascending=False).reset_index()
ordered_grouped_movies_rating

movieId    rating                                              title
                  mean  max count                                   first
0        356  4.164134  5.0   329                     Forrest Gump (1994)
1        318  4.429022  5.0   317        Shawshank Redemption, The (1994)
2        296  4.197068  5.0   307                     Pulp Fiction (1994)
3        593  4.161290  5.0   279        Silence of the Lambs, The (1991)
4       2571  4.192446  5.0   278                      Matrix, The (1999)
...      ...       ...  ...   ...                                     ...
9719    4093  1.500000  1.5     1                              Cop (1988)
9720    4089  2.000000  2.0     1                Born in East L.A. (1987)
9721   58351  4.000000  4.0     1  City of Men (Cidade dos Homens) (2007)
9722    4083  4.000000  4.0     1                      Best Seller (1987)
9723  193609  4.000000  4.0     1     Andrew Dice Clay: Dice Rules (1991)

[9724 rows x 5 columns]

In [ ]:
list_columns = []
for col in ordered_grouped_movies_rating.columns:
    if isinstance(col, tuple):
        list_columns.append("_".join([level for level in col if level]))
    else:
        list_columns.append(col)

In [87]:
ordered_grouped_movies_rating.columns = [
    "_".join([level for level in col if level != ""]) if isinstance(col, tuple) else col
    for col in ordered_grouped_movies_rating.columns
]
ordered_grouped_movies_rating

,movieId,rating_mean,rating_max,rating_count,title_first
0,356,4.164134,5.0,329,Forrest Gump (1994)
1,318,4.429022,5.0,317,"Shawshank Redemption, The (1994)"
2,296,4.197068,5.0,307,Pulp Fiction (1994)
3,593,4.161290,5.0,279,"Silence of the Lambs, The (1991)"
4,2571,4.192446,5.0,278,"Matrix, The (1999)"
...,...,...,...,...,...
9719,4093,1.500000,1.5,1,Cop (1988)
9720,4089,2.000000,2.0,1,Born in East L.A. (1987)
9721,58351,4.000000,4.0,1,City of Men (Cidade dos Homens) (2007)
9722,4083,4.000000,4.0,1,Best Seller (1987)


In [82]:
ordered_grouped_movies_rating.columns

Index(['movieId', 'rating_mean', 'rating_max', 'rating_count', 'title_first'], dtype='object')

In [5]:
ordered_grouped_movies_rating = movies_ratings.groupby("movieId").agg({
        "rating": ["mean", "max", "count"],
        "title": "first"
    }
).sort_values(by=("rating", "mean"), ascending=False).reset_index()

# Flatten MultiIndex column names
ordered_grouped_movies_rating.columns = [
    "_".join([level for level in col if level]) if isinstance(col, tuple) else col
    for col in ordered_grouped_movies_rating.columns
]

# rename title_first to title as title first does not make much sense
ordered_grouped_movies_rating.rename(columns={"title_first": "title"}, inplace=True) 
# inplace here is instructing the rename function to change the in ordered_grouped_movies_rating dataframe instead of returning a new dataframe
ordered_grouped_movies_rating

,movieId,rating_mean,rating_max,rating_count,title
0,88448,5.0,5.0,1,Paper Birds (Pájaros de papel) (2010)
1,100556,5.0,5.0,1,"Act of Killing, The (2012)"
2,143031,5.0,5.0,1,Jump In! (2007)
3,143511,5.0,5.0,1,Human (2015)
4,143559,5.0,5.0,1,L.A. Slasher (2015)
...,...,...,...,...,...
9719,157172,0.5,0.5,1,Wizards of the Lost Kingdom II (1989)
9720,85334,0.5,0.5,1,Hard Ticket to Hawaii (1987)
9721,53453,0.5,0.5,1,Starcrash (a.k.a. Star Crash) (1978)
9722,8494,0.5,0.5,1,"Cincinnati Kid, The (1965)"


In [9]:
mean_rating = ratings["rating"].mean()
ordered_grouped_movies_rating["weighted_mean"] = (ordered_grouped_movies_rating["rating_count"]/(ordered_grouped_movies_rating["rating_count"] + 20)) * ordered_grouped_movies_rating["rating_mean"] + (20 / (ordered_grouped_movies_rating["rating_count"] + 20)) * mean_rating
ordered_grouped_movies_rating.sort_values(by="weighted_mean", ascending=False)

,movieId,rating_mean,rating_max,rating_count,title,weighted_mean
722,318,4.429022,5.0,317,"Shawshank Redemption, The (1994)",4.373980
800,858,4.289062,5.0,192,"Godfather, The (1972)",4.214770
808,2959,4.272936,5.0,218,Fight Club (1999),4.208114
935,260,4.231076,5.0,251,Star Wars: Episode IV - A New Hope (1977),4.177237
933,50,4.237745,5.0,204,"Usual Suspects, The (1995)",4.172014
...,...,...,...,...,...,...
9200,3593,1.657895,4.5,19,Battlefield Earth (2000),2.603363
9096,1499,1.925926,5.0,27,Anaconda (1997),2.596407
9205,1556,1.605263,3.0,19,Speed 2: Cruise Control (1997),2.577722
8538,2701,2.207547,4.5,53,Wild Wild West (1999),2.562070


In [32]:
def get_imdb_rating(movies: pd.DataFrame, ratings: pd.DataFrame, m: int = 10, num_rec: int = 10) -> pd.DataFrame:
    movies_ratings = pd.merge(ratings, movies, on="movieId", how="inner")
    ordered_grouped_movies_rating = movies_ratings.groupby(["movieId", "title"]).agg({
            "rating": ["mean", "count"],
        }
    ).reset_index()

    # Flatten MultiIndex column names
    ordered_grouped_movies_rating.columns = [
        "_".join([level for level in col if level]) if isinstance(col, tuple) else col
        for col in ordered_grouped_movies_rating.columns
    ]

    mean_rating = movies_ratings["rating"].mean() 
    ordered_grouped_movies_rating["weighted_mean"] = (ordered_grouped_movies_rating["rating_count"]/(ordered_grouped_movies_rating["rating_count"] + m)) * ordered_grouped_movies_rating["rating_mean"] + (m / (ordered_grouped_movies_rating["rating_count"] + m)) * mean_rating
    return ordered_grouped_movies_rating.sort_values(by="weighted_mean", ascending=False).head(num_rec)
    

In [33]:
get_imdb_rating(movies=movies,ratings=ratings, m=20, num_rec=15)

,movieId,title,rating_mean,rating_count,weighted_mean
277,318,"Shawshank Redemption, The (1994)",4.429022,317,4.373980
659,858,"Godfather, The (1972)",4.289062,192,4.214770
2224,2959,Fight Club (1999),4.272936,218,4.208114
224,260,Star Wars: Episode IV - A New Hope (1977),4.231076,251,4.177237
46,50,"Usual Suspects, The (1995)",4.237745,204,4.172014
461,527,Schindler's List (1993),4.225000,220,4.164713
921,1221,"Godfather: Part II, The (1974)",4.259690,129,4.157927
257,296,Pulp Fiction (1994),4.197068,307,4.154529
897,1196,Star Wars: Episode V - The Empire Strikes Back...,4.215640,211,4.153814
6693,58559,"Dark Knight, The (2008)",4.238255,149,4.151072


In [8]:
movies_ratings = pd.merge(ratings, movies, on="movieId", how="inner")

In [37]:
movies_ratings.groupby(['movieId', 'title'])['rating'].mean().reset_index()

,movieId,title,rating
0,1,Toy Story (1995),3.920930
1,2,Jumanji (1995),3.431818
2,3,Grumpier Old Men (1995),3.259615
3,4,Waiting to Exhale (1995),2.357143
4,5,Father of the Bride Part II (1995),3.071429
...,...,...,...
9719,193581,Black Butler: Book of the Atlantic (2017),4.000000
9720,193583,No Game No Life: Zero (2017),3.500000
9721,193585,Flint (2017),3.500000
9722,193587,Bungo Stray Dogs: Dead Apple (2018),3.500000


In [38]:
movies_ratings.groupby(['movieId', 'title'])['rating'].count().reset_index()

,movieId,title,rating
0,1,Toy Story (1995),215
1,2,Jumanji (1995),110
2,3,Grumpier Old Men (1995),52
3,4,Waiting to Exhale (1995),7
4,5,Father of the Bride Part II (1995),49
...,...,...,...
9719,193581,Black Butler: Book of the Atlantic (2017),1
9720,193583,No Game No Life: Zero (2017),1
9721,193585,Flint (2017),1
9722,193587,Bungo Stray Dogs: Dead Apple (2018),1


In [9]:
def imbd_rating(movie_rate:pd.DataFrame, m=20)->pd.DataFrame:
    #score=v/(v+m).R +  m/(v+m).C
    #R: average
    #v: count
    R=movie_rate.groupby('movieId')['rating'].mean()
    v=movie_rate.groupby('movieId')['rating'].count()
    C=movie_rate['rating'].mean()
    term1 = (v /(v + m))*R
    term2 = (m/(v +m))*C
    return term1 + term2  

In [47]:
weighted_movie_rate=movies_ratings.groupby(["movieId","title"]).agg({

        "rating": ["count" , "mean"]

    }).reset_index()

In [18]:
weighted_movie_rate=movies_ratings.groupby(["movieId","title"]).agg({

        "rating": ["count" , "mean"]

    }).reset_index()
weighted_movie_rate.columns = [
    "_".join([level for level in col if level]) if isinstance(col, tuple) else col
    for col in weighted_movie_rate.columns
]

In [19]:

#.sort_values(by="rating", ascending=False).reset_index()
m=20
weighted_movie_rate['weighted_rating']=imbd_rating(movies_ratings,m=m)
weighted_movie_rate.sort_values(by='weighted_rating', ascending=False)



,movieId,title,rating_count,rating_mean,weighted_rating
318,360,I Love Trouble (1994),8,2.687500,4.373980
858,1130,"Howling, The (1980)",5,2.900000,4.214770
2959,3972,"Legend of Drunken Master, The (Jui kuen II) (1...",16,4.125000,4.208114
260,300,Quiz Show (1994),81,3.518519,4.177237
50,55,Georgia (1995),1,4.000000,4.172014
...,...,...,...,...,...
9719,193581,Black Butler: Book of the Atlantic (2017),1,4.000000,NaN
9720,193583,No Game No Life: Zero (2017),1,3.500000,NaN
9721,193585,Flint (2017),1,3.500000,NaN
9722,193587,Bungo Stray Dogs: Dead Apple (2018),1,3.500000,NaN


In [16]:
weighted_movie_rate

,movieId,title,rating_count,rating_mean,weighted_rating
318,360,I Love Trouble (1994),8,2.687500,4.373980
858,1130,"Howling, The (1980)",5,2.900000,4.214770
2959,3972,"Legend of Drunken Master, The (Jui kuen II) (1...",16,4.125000,4.208114
260,300,Quiz Show (1994),81,3.518519,4.177237
50,55,Georgia (1995),1,4.000000,4.172014
...,...,...,...,...,...
9719,193581,Black Butler: Book of the Atlantic (2017),1,4.000000,NaN
9720,193583,No Game No Life: Zero (2017),1,3.500000,NaN
9721,193585,Flint (2017),1,3.500000,NaN
9722,193587,Bungo Stray Dogs: Dead Apple (2018),1,3.500000,NaN


In [ ]:
def get_n_highest_rating(movie_ratings: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    average_rating_movies = movie_ratings.groupby("movieId")["rating"].mean().reset_index()

In [17]:
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ','.join(x)).reset_index()
tags_grouped.loc[tags_grouped["movieId"] == 60756,:]

,movieId,tag
1407,60756,"funny,Highly quotable,will ferrell,comedy,funn..."


## Popularity-Based Recommendation

In [ ]:
def weighted_popularity_recommendation(n: int=10):
    current_date = datetime.now()
    ratings['days_since_rating'] = (current_date - ratings['timestamp']).dt.days
    ratings['time_weight'] = 1 / (1 + ratings['days_since_rating'])
    ratings['weighted_rating'] = ratings['rating'] * ratings['time_weight']

    weighted_avg = ratings.groupby('movieId')['weighted_rating'].mean().reset_index()
    top_movies = weighted_avg.sort_values(by='weighted_rating', ascending=False).head(n)
    return pd.merge(top_movies, movies, on='movieId')[['title', 'weighted_rating']]

weighted_popularity_recommendation()


,title,weighted_rating
0,Tickling Giants (2017),0.001893
1,A Detective Story (2003),0.001891
2,Saving Face (2004),0.001883
3,Blue Planet II (2017),0.001867
4,Kung Fu Panda: Secrets of the Masters (2011),0.001835
5,Won't You Be My Neighbor? (2018),0.001831
6,Winnie the Pooh Goes Visiting (1971),0.001806
7,Investigation Held by Kolobki (1986),0.001806
8,Karlson Returns (1970),0.001806
9,Vacations in Prostokvashino (1980),0.001806


# Collaborative filtering memory Based 

## Simple Memory based

In [ ]:
def user_based_recommendation(user_id, k=5):
    # Create user–item rating matrix
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

    # Compute similarity between users
    similarity = cosine_similarity(user_item_matrix)

    # Convert to DataFrame for easier indexing
    similarity_df = pd.DataFrame(similarity,
                                 index=user_item_matrix.index,
                                 columns=user_item_matrix.index)

    # Find top-k users most similar to the target user (exclude user themself)
    similar_users = similarity_df[user_id].sort_values(ascending=False).iloc[1:k+1].index

    # Average ratings of the similar users to generate recommendation scores
    recommendations = user_item_matrix.loc[similar_users].mean().sort_values(ascending=False).index

    # Return the top 10 movie titles
    return movies[movies['movieId'].isin(recommendations)]['title'].head(10)

In [27]:
def item_based_recommendation(user_id):
    # Create user–item rating matrix
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

    # Compute similarity between items (movies)
    similarity = cosine_similarity(user_item_matrix.T)

    # Convert to DataFrame for easier indexing
    similarity_df = pd.DataFrame(similarity,
                                 index=user_item_matrix.columns,
                                 columns=user_item_matrix.columns)

    # Get the user's ratings
    user_movies = user_item_matrix.loc[user_id]

    # Identify movies the user has not watched yet
    unseen_movies = user_movies[user_movies == 0].index

    # Rank unseen movies by their average similarity to all other movies
    recommendations = similarity_df.loc[unseen_movies].mean(axis=1).sort_values(ascending=False).index

    # Return the top 10 movie titles
    return movies[movies['movieId'].isin(recommendations)]['title'].head(10)

item_based_recommendation(user_id=2)

0                      Toy Story (1995)
1                        Jumanji (1995)
2               Grumpier Old Men (1995)
3              Waiting to Exhale (1995)
4    Father of the Bride Part II (1995)
5                           Heat (1995)
6                        Sabrina (1995)
7                   Tom and Huck (1995)
8                   Sudden Death (1995)
9                      GoldenEye (1995)
Name: title, dtype: object

## Memory based with weighting

In [ ]:
def user_based_collaborative_filtering(user_id, k=5, n=10):
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
    user_similarity = cosine_similarity(user_item_matrix)
    similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

    similar_users = similarity_df[user_id].sort_values(ascending=False).iloc[1:k+1].index
    user_ratings = user_item_matrix.loc[user_id]
    unseen = user_ratings[user_ratings == 0].index

    predictions = {
        movie_id: sum(similarity_df[user_id][sim] * user_item_matrix.loc[sim, movie_id] for sim in similar_users) / 
                  sum(similarity_df[user_id][sim] for sim in similar_users)
        for movie_id in unseen
    }
    top = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n]
    return movies[movies['movieId'].isin([m[0] for m in top])][['title']]

user_based_collaborative_filtering(user_id=1)


In [ ]:
def item_based_collaborative_filtering(user_id, n=10):
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
    item_similarity = cosine_similarity(user_item_matrix.T)
    sim_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    user_ratings = user_item_matrix.loc[user_id]
    rated = user_ratings[user_ratings > 0]

    predictions = {}
    for movie_id in user_item_matrix.columns:
        if movie_id not in rated.index:
            sim_scores = [sim_df[movie_id][other] * rated[other] for other in rated.index]
            sim_sum = sum(sim_df[movie_id][other] for other in rated.index)
            predictions[movie_id] = sum(sim_scores) / sim_sum if sim_sum else 0

    top = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n]
    return movies[movies['movieId'].isin([m[0] for m in top])][['title']]

item_based_collaborative_filtering(user_id=1)


# Collaborative filtering model based

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

def evaluate_svd_rmse(ratings, test_size=0.2, random_state=42):
    reader = Reader(rating_scale=(ratings.rating.min(), ratings.rating.max()))
    data = Dataset.load_from_df(
        ratings[['userId', 'movieId', 'rating']], reader
    )

    trainset, testset = train_test_split(
        data, test_size=test_size, random_state=random_state
    )

    model = SVD(random_state=random_state)
    model.fit(trainset)

    predictions = model.test(testset)
    rmse = accuracy.rmse(predictions)

    return model, rmse

def svd_top_n_with_threshold(
    model,
    ratings,
    movies,
    user_id,
    n=10,
    min_rating=4.0
):
    all_movies = ratings['movieId'].unique()
    rated_movies = ratings[ratings['userId'] == user_id]['movieId']
    unseen_movies = set(all_movies) - set(rated_movies)

    predictions = []
    for movie_id in unseen_movies:
        est = model.predict(user_id, movie_id).est
        if est >= min_rating:
            predictions.append((movie_id, est))

    top_n = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]

    return movies[
        movies['movieId'].isin([m[0] for m in top_n])
    ][['title']]

In [ ]:
def svd_top_n_with_threshold(
    model,
    ratings,
    movies,
    user_id,
    n=10,
    min_rating=4.0
):
    all_movies = ratings['movieId'].unique()
    rated_movies = ratings[ratings['userId'] == user_id]['movieId']
    unseen_movies = set(all_movies) - set(rated_movies)

    predictions = []
    for movie_id in unseen_movies:
        est = model.predict(user_id, movie_id).est
        if est >= min_rating:
            predictions.append((movie_id, est))

    top_n = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]

    return movies[
        movies['movieId'].isin([m[0] for m in top_n])
    ][['title']]

## Content-Based Filtering (Genres + Tags)

In [25]:
def tag_based_recommendation(movie_id, n=10):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies['combined_features'])
    similarity = cosine_similarity(tfidf_matrix)

    idx = movies[movies['movieId'] == movie_id].index[0]
    similar = similarity[idx].argsort()[::-1][1:n+1]
    return movies.iloc[similar][['title', 'genres', 'tag']]

tag_based_recommendation(movie_id=1)


,title,genres,tag
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,Pixar
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,"animation,Disney,funny,original,Pixar,sequel,T..."
8695,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi,fun
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,
8927,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,


In [26]:
movies[movies["movieId"] == 1]

,movieId,title,genres,tag,combined_features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"pixar,pixar,fun",Adventure|Animation|Children|Comedy|Fantasy pi...


## 🔀 Hybrid Recommendation System

In [ ]:
def hybrid_recommendation(user_id, movie_id, weights=(0.5, 0.5), n=10):
    content_df = tag_based_recommendation(movie_id, n)
    content_df['score'] = weights[0] / (1 + content_df.index)

    collab_df = user_based_collaborative_filtering(user_id, n=n)
    collab_df = pd.DataFrame({'title': collab_df['title'], 'score': [weights[1] / (i + 1) for i in range(len(collab_df))]})

    combined = pd.concat([content_df[['title', 'score']], collab_df])
    return combined.groupby('title').agg({'score': 'sum'}).reset_index().sort_values(by='score', ascending=False).head(n)

hybrid_recommendation(user_id=1, movie_id=1)


## Cold Start Recommendation

In [6]:
def cold_start_recommendation(preferred_genres, n=10):
    filtered = movies[movies['genres'].str.contains('|'.join(preferred_genres))]
    avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
    merged = pd.merge(filtered, avg_ratings, on='movieId')
    return merged.sort_values(by='rating', ascending=False)[['title', 'genres', 'rating']].head(n)

cold_start_recommendation(['Adventure', 'Fantasy'])


,title,genres,rating
1279,Idiots and Angels (2008),Animation|Drama|Fantasy,5.0
1597,Dragons: Gift of the Night Fury (2011),Adventure|Animation|Comedy,5.0
1366,My Left Eye Sees Ghosts (Ngo joh aan gin diy g...,Comedy|Fantasy|Romance,5.0
1365,Holy Motors (2012),Drama|Fantasy|Musical|Mystery|Sci-Fi,5.0
1576,Return to Treasure Island (1988),Adventure|Animation|Comedy,5.0
1364,"Odd Life of Timothy Green, The (2012)",Comedy|Drama|Fantasy,5.0
1584,12 Chairs (1976),Adventure|Comedy,5.0
1197,Mickey's Once Upon a Christmas (1999),Animation|Comedy|Fantasy,5.0
1593,L.A. Slasher (2015),Comedy|Crime|Fantasy,5.0
1357,Goodbye Charlie (1964),Comedy|Fantasy|Romance,5.0


## Genre Preference Filtering

In [20]:
def genre_preference_recommendation(user_id, preferred_genres, n=10):
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    filtered = movies[movies['genres'].str.contains('|'.join(preferred_genres)) & ~movies['movieId'].isin(user_rated)]
    avg = ratings.groupby('movieId')['rating'].mean().reset_index()
    recs = pd.merge(filtered, avg, on='movieId').sort_values(by='rating', ascending=False)
    return recs[['title', 'genres', 'rating']].head(n)

genre_preference_recommendation(user_id=1, preferred_genres=['Comedy', 'Romance'])


,title,genres,rating
3586,Runaway Brain (1995),Animation|Comedy|Sci-Fi,5.0
3148,Strictly Sexual (2008),Comedy|Drama|Romance,5.0
4067,Battle For Sevastopol (2015),Drama|Romance|War,5.0
1213,Duel in the Sun (1946),Drama|Romance|Western,5.0
4060,George Carlin: Jammin' in New York (1992),Comedy,5.0
4058,Hollywood Chainsaw Hookers (1988),Comedy|Horror,5.0
2936,"Valet, The (La doublure) (2006)",Comedy,5.0
4050,Return to Treasure Island (1988),Adventure|Animation|Comedy,5.0
3632,English Vinglish (2012),Comedy|Drama,5.0
2931,Sun Alley (Sonnenallee) (1999),Comedy|Romance,5.0


In [22]:
movies[movies['genres'].str.contains('|'.join(["Comedy", "Romance"]))]

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
6,7,Sabrina (1995),Comedy|Romance
...,...,...,...
9732,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi
9734,193571,Silver Spoon (2014),Comedy|Drama
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy


## Diversity in Recommendations

In [ ]:
def diverse_recommendation(user_id, n=10):
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    candidates = movies[~movies['movieId'].isin(user_rated)]
    avg_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
    candidates = pd.merge(candidates, avg_ratings, on='movieId').sort_values(by='rating', ascending=False)
    diverse = candidates.groupby('genres').head(1).head(n)
    return diverse[['title', 'genres', 'rating']]

diverse_recommendation(user_id=1)


In [21]:
"|".join(["comedy", "romance"])

'comedy|romance'